# 0002 --- Chunk decoding: a tutorial

Companion to the *Chunk decoding for large-volume latent diffusion* talk. The talk gives the picture; this notebook lets you reproduce every figure and inspect every intermediate quantity. Audience is the same: ML engineers who want to understand the algorithm well enough to modify, verify, or replace pieces of it.

**What chunk decoding solves.** The VAE decoder of a latent-diffusion model maps a latent of spatial shape $(L/F)^d$ to a pixel volume of shape $L^d$ ($F=8$ for our 3D production VAE). At $L=2304$ a single full-resolution intermediate is several TB of float32. We want a one-pass decoding scheme whose peak GPU memory is independent of $L$.

**What chunk decoding is.** Tiled inference. Partition the latent into tiles; for each tile read it together with its halo, run the decoder, write the cropped centre to a CPU output buffer; stitch. Two complications relative to vanilla overlap-tile inference: (i) the decoder upsamples by $2$ several times, so the geometry must propagate the tile through multiple resolutions; (ii) GroupNorm has *infinite* receptive field, which we patch with a first-tile cache.

**Code lives in** `diffsci2.extra.chunk_decode_2`. Production entry point: `chunk_decode_strategy_b` (single GPU), `chunk_decode_parallel` (multi-GPU).

**Self-contained.** All assets are in `0002-data/` next to this notebook (toy 2D VAE checkpoint, three Berea slices). Every figure is generated inline.

## 0. Setup

Imports, paths to local assets, and the toy 2D VAE (Berea-trained, `ch_mult=[1,2,4]`, $F=4$). Same architectural shape as the production 3D VAE, smaller depths so every figure stays 2D.

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os
import numpy as np
import torch
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from IPython.display import display

import diffsci2.nets
import diffsci2.models
from diffsci2.extra import chunk_decode_2 as cd2

DEVICE = 'cuda:0' if torch.cuda.is_available() else 'cpu'
torch.manual_seed(0); np.random.seed(0)

# Resolve 0002-data/ relative to wherever the notebook is launched from.
for cand in ['0002-data',
             'notebooks/tutorials/0002-data',
             '/home/ubuntu/repos/DiffSci2/notebooks/tutorials/0002-data']:
    if os.path.isdir(cand):
        DATA_DIR = os.path.abspath(cand); break
else:
    raise FileNotFoundError('cannot find 0002-data/ next to the notebook')
VAE_CKPT  = os.path.join(DATA_DIR, 'vae.ckpt')
BEREA_NPY = os.path.join(DATA_DIR, 'berea_slices.npy')
print('DATA_DIR =', DATA_DIR)

In [ ]:
# Toy 2D VAE used throughout the talk and this notebook.
vae_cfg = diffsci2.nets.VAENetConfig(
    dimension=2, in_channels=1, out_channels=1, z_channels=4, z_dim=4,
    ch=32, ch_mult=[1, 2, 4], num_res_blocks=2, attn_resolutions=[],
    dropout=0.0, resolution=128, has_mid_attn=False, resamp_with_conv=True,
    attn_type='vanilla', tanh_out=False, input_bias=True, output_bias=True,
    with_time_emb=False, double_z=True, num_groups=32, patch_size=None,
    memory_efficient_variant=False, use_flash_attention=True, minimal_rf_mode=False,
)
vae_net = diffsci2.nets.VAENet(config=vae_cfg)
vae = diffsci2.models.VAEModule.load_from_checkpoint(
    VAE_CKPT, config=diffsci2.models.VAEModuleConfig(), encdec=vae_net,
).to(DEVICE).eval()
decoder = vae.decoder    # the VAEDecoder object expected by chunk_decode_2

# Three Berea slices (z = 200, 400, 600), 512x512 each, uint8.
berea_slices = np.load(BEREA_NPY).astype(np.float32)
berea_slice  = berea_slices[1]   # the z=400 slice; reused throughout

print(f'decoder: {type(decoder).__name__}'
      f' | num_resolutions = {vae_cfg.num_resolutions}'
      f' | F = {2 ** (vae_cfg.num_resolutions - 1)}')
print(f'berea_slices: shape={berea_slices.shape}, dtype={berea_slices.dtype}')

In [ ]:
# Three Berea slices --- the running example.
fig, axes = plt.subplots(1, 3, figsize=(12, 4.2))
for ax, (z, sl) in zip(axes, zip([200, 400, 600], berea_slices)):
    ax.imshow(sl, cmap='gray')
    ax.set_xticks([]); ax.set_yticks([])
    ax.set_title(f'slice z = {z}\nporosity = {1 - sl.mean():.3f}')
fig.suptitle('Berea sandstone (1000$^3$ voxels @ 2.25 $\\mu$m)', fontsize=12, y=1.02)
fig.tight_layout(); plt.show()

## 1. The decoding bottleneck

Production VAE: `ch=32`, `ch_mult=[1,2,4,4]`, $F=8$. The intermediate at full resolution has $C_{\max}=32$ channels; in float32 a single full-res tensor is
$$
4 \cdot 32 \cdot L^3 \;\;\Rightarrow\;\; L=1024:\;128\text{ GB},\quad L=2048:\;{\sim}\,1\text{ TB},\quad L=4096:\;{\sim}\,8\text{ TB}.
$$
Way past any single GPU.

Chunked peak is the size of *one tile's* activations, roughly $C_{\max}\cdot(T+2 r_{\max})^d$ in float32 with $T$ the tile size at output scale and $r_{\max}$ the largest per-stage halo. The output cap `max_stage_out_chunk` (default 128 px) puts a hard ceiling on $T$. Both numbers are independent of $L$.

In [ ]:
# Memory scaling: monolithic vs chunked peak GPU memory for the production 3D VAE.
GPU_RAM_GB = 40                # the GPU we actually use
C_full = 32                    # production VAE: 32 channels at full resolution

def monolithic_peak_GB(L):
    return 4 * C_full * (L ** 3) / 2**30

def chunked_peak_GB(max_out=128, dst_max=8, C_dst=64, delta_r_max=7, overhead=3.0):
    side = max_out + 2 * delta_r_max * dst_max
    return overhead * C_dst * (side ** 3) * 4 / 2**30

Ls_pix = np.array([256, 384, 512, 768, 1024, 1280, 1536, 2048, 2304, 3072])
mono   = np.array([monolithic_peak_GB(L) for L in Ls_pix])
chunk  = np.full_like(Ls_pix, chunked_peak_GB(), dtype=float)

fig, ax = plt.subplots(figsize=(7.2, 4.4))
ax.plot(Ls_pix, mono,  'o-', color='C3', lw=2, label='monolithic peak (single full-res tensor)')
ax.plot(Ls_pix, chunk, 's-', color='C0', lw=2, label='chunked peak (chunk_latent = 40)')
ax.axhline(GPU_RAM_GB, ls='--', color='black', lw=1)
ax.text(Ls_pix[1], GPU_RAM_GB * 1.18, f'{GPU_RAM_GB} GB GPU', fontsize=10)
ax.set_xlabel('output side $L$ (pixels)')
ax.set_ylabel('peak GPU activation memory (GB, log)')
ax.set_xscale('log'); ax.set_yscale('log')
ax.set_title('Peak GPU activation memory: monolithic vs chunked  (production 3D VAE)', fontsize=11)
ax.legend(loc='upper left'); ax.grid(True, which='both', alpha=0.3)
fig.tight_layout(); plt.show()

## 2. The algorithm in one figure

This is the load-bearing diagram of the whole approach. **Top row**: the latent (input). **Bottom row**: the output buffer (pixel space). Both at their own scale.

For each step:
1. Pick the next tile (red solid in the latent).
2. Read tile + halo (red dashed) from the latent.
3. Run the decoder.
4. Crop the valid centre.
5. Write to the corresponding position in the output buffer (red solid in the output).

After 4 steps the entire $256\times 256$ output is filled. The rest of this notebook formalises every word of those five lines: "halo", "valid centre", "corresponding position", and what happens when there are several upsampling stages between latent and output.

In [ ]:
# THE centerpiece. Latent partitioned 2x2 = 4 tiles; one column per tile-step.
patch = torch.from_numpy(berea_slice[:256, :256][None, None]).to(DEVICE).float()
with torch.no_grad():
    z = vae.encode(patch, sample=False)['zsample']
    x_full = vae.decode(z, postdecode=False).cpu().numpy()[0, 0]
z_ch0 = z.cpu().numpy()[0, 0]

F = 4
L_lat = z.shape[-1]                         # 64
chunk_latent = 42                           # valid = 32 -> 2 spans of 32
r0 = 5
spans = cd2.make_center_spans_1d(L_lat, chunk_latent, r0)
tiles = [(i, j) for i in range(2) for j in range(2)]
vmin, vmax = float(x_full.min()), float(x_full.max())
bg_pix = vmin + 0.85 * (vmax - vmin)

fig_step, axes = plt.subplots(2, 4, figsize=(15.5, 7.6))
for col, (ti, tj) in enumerate(tiles):
    out_buf = np.full_like(x_full, np.nan)
    for k in range(col + 1):
        a, b = tiles[k]; hsa, hea = spans[a]; wsa, wea = spans[b]
        out_buf[hsa*F:hea*F, wsa*F:wea*F] = x_full[hsa*F:hea*F, wsa*F:wea*F]
    hs, he = spans[ti]; ws, we = spans[tj]

    # top: latent with current chunk + halo
    ax = axes[0, col]
    ax.imshow(z_ch0, cmap='viridis', extent=[-0.5, L_lat-0.5, L_lat-0.5, -0.5])
    rh0 = max(0, hs-r0); rh1 = min(L_lat, he+r0)
    rw0 = max(0, ws-r0); rw1 = min(L_lat, we+r0)
    ax.add_patch(mpatches.Rectangle((rw0-0.5, rh0-0.5), rw1-rw0, rh1-rh0,
                                     fill=False, edgecolor='red', lw=1.8, ls='--'))
    ax.add_patch(mpatches.Rectangle((ws-0.5, hs-0.5), we-ws, he-hs,
                                     fill=False, edgecolor='red', lw=2.6))
    ax.set_title(f'step {col+1}/4  (chunk at tile ({ti},{tj}))', fontsize=11)
    ax.set_xticks([]); ax.set_yticks([])
    if col == 0:
        ax.set_ylabel('LATENT  $z_0$\n64$\\times$64', rotation=0, ha='right',
                      va='center', labelpad=22, fontsize=11)

    # bottom: output buffer accumulating
    ax = axes[1, col]
    disp = np.where(np.isnan(out_buf), bg_pix, out_buf)
    ax.imshow(disp, cmap='gray', vmin=vmin, vmax=vmax)
    ax.add_patch(mpatches.Rectangle((ws*F-0.5, hs*F-0.5), (we-ws)*F, (he-hs)*F,
                                     fill=False, edgecolor='red', lw=2.6))
    ax.set_title(f'output: {col+1}/4 tiles filled', fontsize=11)
    ax.set_xticks([]); ax.set_yticks([])
    if col == 0:
        ax.set_ylabel('OUTPUT  $\\hat{x}$\n256$\\times$256', rotation=0, ha='right',
                      va='center', labelpad=22, fontsize=11)

fig_step.suptitle('Chunk decoding: chunk moves through the latent (top) $\\to$ '
                  'corresponding tile is written into the output (bottom). '
                  'Solid red = current tile. Dashed red = halo. Grey = not yet filled.',
                  fontsize=12, y=1.00)
fig_step.tight_layout(rect=[0, 0, 1, 0.96])
plt.show()

## 3. The VAE pipeline

Standard latent-diffusion VAE. $F=4$ for the toy, $F=8$ in production. The latent has 4 channels at $L/F$ per axis. The diffusion model lives entirely in latent space; the only place we hit full resolution is the final decode. *That decode is what we tile.*

In [ ]:
# Live encode / decode demo with shapes.
patch = torch.from_numpy(berea_slice[:256, :256][None, None]).to(DEVICE).float()
with torch.no_grad():
    z = vae.encode(patch, sample=False)['zsample']
    x_recon = vae.decode(z, postdecode=False)
print(f'patch:    {tuple(patch.shape)}    {patch.dtype}')
print(f'latent z: {tuple(z.shape)}    {z.dtype}')
print(f'recon:    {tuple(x_recon.shape)}    {x_recon.dtype}')

In [ ]:
# VAE pipeline figure: input -> 4 latent channels -> reconstruction.
z_np = z.cpu().numpy()[0]
x_np = x_recon.cpu().numpy()[0, 0]

fig, axes = plt.subplots(1, 8, figsize=(16.5, 4.2),
    gridspec_kw={'width_ratios': [3, 0.4, 1, 1, 1, 1, 0.4, 3], 'wspace': 0.15})
axes[0].imshow(patch[0, 0].cpu().numpy(), cmap='gray')
axes[0].set_title('input  $x$\n256$\\times$256')

axes[1].set_axis_off()
axes[1].text(0.5, 0.55, 'Enc', ha='center', va='bottom', fontsize=12,
             transform=axes[1].transAxes)
axes[1].annotate('', xy=(0.95, 0.5), xytext=(0.05, 0.5),
                 xycoords='axes fraction', textcoords='axes fraction',
                 arrowprops=dict(arrowstyle='-|>', lw=2, color='black'))
for c in range(4):
    axes[2+c].imshow(z_np[c], cmap='viridis')
    axes[2+c].set_title(f'latent  $z_{c}$\n64$\\times$64')
axes[6].set_axis_off()
axes[6].text(0.5, 0.55, 'Dec', ha='center', va='bottom', fontsize=12,
             transform=axes[6].transAxes)
axes[6].annotate('', xy=(0.95, 0.5), xytext=(0.05, 0.5),
                 xycoords='axes fraction', textcoords='axes fraction',
                 arrowprops=dict(arrowstyle='-|>', lw=2, color='black'))
axes[7].imshow(x_np, cmap='gray')
axes[7].set_title('reconstruction  $\\hat{x}$\n256$\\times$256')
for i in [0, 2, 3, 4, 5, 7]:
    axes[i].set_xticks([]); axes[i].set_yticks([])
fig.suptitle('VAE encoder / 4-channel latent / decoder on a Berea patch',
             fontsize=12, y=1.02)
fig.tight_layout(); plt.show()

## 4. Halos

**Halo (definition).** For a tile $\Omega' \subseteq \Omega$ and a radius $r$, the *halo* of $\Omega'$ at radius $r$ is the boundary band
$$
\Omega'^{+r}\setminus \Omega', \qquad
\Omega'^{+r} = \bigl\{\,p \in \mathbb{Z}^d : \exists\, q \in \Omega',\;\|p-q\|_{\infty}\le r\,\bigr\}.
$$

Three halo radii on the same $8\times 8$ tile inside a $32\times 32$ domain:

In [ ]:
# Halo panel: same tile, three radii.
L = 32; tile_origin = (12, 12); tile_size = 8; halos = (2, 5, 8)
fig, axes = plt.subplots(1, len(halos), figsize=(13, 4))
for ax, r in zip(axes, halos):
    ax.set_xlim(-0.5, L - 0.5); ax.set_ylim(L - 0.5, -0.5)
    ax.set_aspect('equal'); ax.set_xticks([]); ax.set_yticks([])
    ax.set_title(rf'halo radius $r$ = {r}')
    ax.add_patch(mpatches.Rectangle((-0.5, -0.5), L, L, fill=False, ec='black', lw=1))
    i0, j0 = tile_origin; t = tile_size
    ax.add_patch(mpatches.Rectangle(
        (j0 - r - 0.5, i0 - r - 0.5), t + 2*r, t + 2*r,
        facecolor='#ffd9b3', edgecolor='#cc5500', lw=2, alpha=0.8,
        label=r"read window  $\Omega'^{+r}$"))
    ax.add_patch(mpatches.Rectangle(
        (j0 - 0.5, i0 - 0.5), t, t,
        facecolor='#a6d96a', edgecolor='#1a9850', lw=2,
        label=r"valid tile  $\Omega'$"))
    for k in range(L + 1):
        ax.plot([k-0.5, k-0.5], [-0.5, L-0.5], color='gray', lw=0.2)
        ax.plot([-0.5, L-0.5], [k-0.5, k-0.5], color='gray', lw=0.2)
    ax.legend(loc='lower right', fontsize=8, framealpha=0.95)
fig.suptitle(r"A tile $\Omega'$ and its halo $\Omega'^{+r}\!\setminus\!\Omega'$  on a 32$\times$32 latent grid",
             fontsize=12)
fig.tight_layout(); plt.show()

Re-read the centerpiece figure with that vocabulary in hand. The dashed red region is the halo. The solid red is the valid centre we keep. *That is the whole algorithm.*

What's left is to figure out what radius $r$ a given operator needs --- and how to handle the $\times 2$ upsamplings inside the decoder.

In [ ]:
# Re-display the centerpiece figure (saved as fig_step in cell 9).
display(fig_step)

## 5. Effective receptive radius

**Definition.** $f$ has *effective receptive radius* $r$ iff for every $\Omega' \subseteq \Omega$ with $\Omega'^{+r}\subseteq \Omega$,
$$
\left.f(x)\right|_{\Omega'} \;=\; \left.f\bigl(\left.x\right|_{\Omega'^{+r}}\bigr)\right|_{\Omega'}. \qquad (\mathrm{RF})
$$
Equivalently, $f$'s output on $\Omega'$ depends only on the input restricted to $\Omega'^{+r}$. Identity (RF) is what makes tiled inference *exact*.

**Operators we care about.**

| Operator | $r$ |
| --- | --- |
| pointwise nonlinearity (ReLU, SiLU, ...) | 0 |
| channel LayerNorm, RMSNorm, instance-on-pixel | 0 |
| $3{\times}3$ convolution | 1 |
| $5{\times}5$ convolution | 2 |
| nearest-neighbour upsampling | 0 |
| attention | $\infty$ |
| GroupNorm | $\infty$ (statistics over all spatial positions) |

**Composition rule.** $r(f \circ g) = r(f) + r(g)$.

**Skip / residual rule.** $r(f + g) = \max\bigl(r(f), r(g)\bigr)$. A standard residual block $x \mapsto x + g(x)$ therefore has $r = r(g)$ (the skip branch contributes 0).

**Consequence.** For a network without attention and without globally-aware normalisation, $r$ is *exactly* computable from the architecture --- you walk the layers.

In [ ]:
# Empirical sanity check on the composition rule. Stack k 3x3 convs (random weights);
# r should be k. Probe by perturbing one input pixel and checking how far the
# perturbation propagates in the output.
@torch.no_grad()
def empirical_rf(net, L=64, eps=1.0):
    x0 = torch.zeros(1, 1, L, L)
    y0 = net(x0)
    x1 = x0.clone(); x1[0, 0, L//2, L//2] = eps
    y1 = net(x1)
    diff = (y1 - y0).abs()[0, 0]
    nz = diff.nonzero(as_tuple=False)
    if nz.numel() == 0: return 0
    cy, cx = L//2, L//2
    return int(max((nz[:, 0] - cy).abs().max(), (nz[:, 1] - cx).abs().max()))

for k in [1, 2, 3, 5, 8]:
    net = torch.nn.Sequential(*[
        torch.nn.Conv2d(1, 1, 3, padding=1, bias=False) for _ in range(k)
    ])
    for m in net.modules():
        if isinstance(m, torch.nn.Conv2d):
            torch.nn.init.normal_(m.weight, std=0.5)
    print(f'  k={k} convs:  empirical r = {empirical_rf(net)}   (expected {k})')

**Receptive field of the toy decoder.** The decoder satisfies (RF) layer by layer (no attention, GroupNorm handled separately). `VAEDecoder.calculate_receptive_field()` walks the layers using the rules above:

In [ ]:
info = decoder.calculate_receptive_field()
print(f"rf_latent       = {info['rf_latent']}   (full-decoder RF, latent units)")
print(f"rf_after_middle = {info['rf_after_middle']}   (after stage 0)")
print(f"rf_per_block    = {info['rf_per_block']}   (per ResBlock contribution)")
print(f"spatial_factor  = {info['spatial_upsampling_factor']}")
print('\nLayer-by-layer trace:')
for line in info['trace']:
    print('   ', line)

**Empirical (RF) check on stage 0.** Pass a random latent through stage 0 globally and on a tile + halo, then compare the cropped centre. Below the stage-0 RF radius (5) the local computation is wrong; above it the convolutional contribution to the discrepancy collapses. The small residual ($\sim 10^{-1}$) at $r\in\{5,6\}$ is the GroupNorm contribution --- different statistics on different inputs --- which doesn't disappear by enlarging $r$. Fixed in §10.

In [ ]:
@torch.inference_mode()
def stage0_diff(decoder, L=24, tile=(slice(8, 16), slice(8, 16)), r=5, seed=0):
    g = torch.Generator(device=DEVICE).manual_seed(seed)
    z_ = torch.randn(1, vae_cfg.z_dim, L, L, generator=g, device=DEVICE)
    y_global = cd2.run_vae_stage0(decoder, z_, temb=None)
    h_slice = slice(max(0, tile[0].start - r), min(L, tile[0].stop + r))
    w_slice = slice(max(0, tile[1].start - r), min(L, tile[1].stop + r))
    y_local = cd2.run_vae_stage0(decoder, z_[:, :, h_slice, w_slice], temb=None)
    crop_h = slice(tile[0].start - h_slice.start, tile[0].stop - h_slice.start)
    crop_w = slice(tile[1].start - w_slice.start, tile[1].stop - w_slice.start)
    return (y_local[:, :, crop_h, crop_w] - y_global[:, :, tile[0], tile[1]]).abs().max().item()

for r in range(0, 11):
    print(f'  r={r:2d}:  max|local - global|  =  {stage0_diff(decoder, r=r):.3e}')

**Cumulative receptive field of the whole decoder.** Walking the layers with the composition + skip rules:

In [ ]:
# Bar chart of cumulative RF, stage-coloured.
labels, rfs = [], []
for line in info['trace']:
    if 'no RF change' in line: continue
    name, _, rest = line.partition(':')
    labels.append(name.strip())
    rfs.append(int(rest.strip().split('=')[-1]))

fig, ax = plt.subplots(figsize=(9, 3.6))
xs = np.arange(len(labels))
colors = []
for l in labels:
    if 'mid' in l:                            colors.append('#a6cee3')   # S0
    elif 'up[2]' in l or 'up[1]' in l:        colors.append('#fdae61')   # S1, S2
    elif 'up[0]' in l or 'conv_out' in l:     colors.append('#a6d96a')   # S3
    else:                                     colors.append('#cccccc')
ax.bar(xs, rfs, color=colors, edgecolor='black', lw=0.5)
for x, rf in zip(xs, rfs):
    ax.text(x, rf + 0.6, str(rf), ha='center', va='bottom', fontsize=9)
ax.set_xticks(xs); ax.set_xticklabels(labels, rotation=35, ha='right')
ax.set_ylabel('cumulative RF (latent units)')
ax.set_title('Receptive field grows from 1 to 49 across the toy decoder', fontsize=12)
ax.set_ylim(0, max(rfs) * 1.18); ax.grid(axis='y', alpha=0.3)
fig.tight_layout(); plt.show()

The cumulative radius reaches **24 latent cells**. At $F=4$ that is **96 pixels of halo per side**, larger than most tiles --- treating the whole decoder as one black-box operator and tiling its read window at output scale is not viable.

## 6. Stage decomposition

**Idea.** Split the decoder into stages whose halos are paid in *latent* units, with the intermediate stage outputs spilled to CPU. The GPU only ever sees one tile of the *current* stage.

**Stages.**
- **S0** --- post-quant + conv-in + middle ResBlocks (latent resolution).
- **S1, ..., S$_{N-1}$** --- one upsampling level each: ResBlocks then $\times 2$ upsample.
- **S$_N$** --- final upsampling level (no upsample) + norm + conv-out.

In [ ]:
# Per-stage geometry.
radii, scales = cd2.compute_stage_radii_and_scales(decoder)
deltas = cd2.compute_delta_radii(radii)
num_stages = vae_cfg.num_resolutions + 1
print(f'  num_stages    = {num_stages}')
print(f'  radii (lat)   = {radii}     # cumulative halo after each stage')
print(f'  delta_r       = {deltas}    # halo *added* by each stage')
print(f'  scales after  = {scales}    # output scale relative to the latent grid')
print()
print(f"  {'stage':<6}{'role':<22}{'delta_r':>9}{'src':>6}{'dst':>6}{'k':>4}")
roles = ['post_q+in+mid', 'up[N-1]+up2x', 'up[N-2]+up2x', 'up[0]+norm+out']
for s in range(num_stages):
    src = scales[s-1] if s > 0 else 1
    print(f'  S{s:<5}{roles[s]:<22}{deltas[s]:>9}{src:>6}{scales[s]:>6}{scales[s]//src:>4}')

In [ ]:
# Decoder anatomy diagram. Layers laid out left-to-right, coloured by stage.
layers = [
    ('post_quant\n1$\\times$1',   0, 4,   1, ''),
    ('conv_in\n3$\\times$3',      0, 128, 1, ''),
    ('mid.block_1\nResBlock',      0, 128, 1, ''),
    ('mid.block_2\nResBlock',      0, 128, 1, ''),
    ('up[2]\n3$\\times$ ResBlock', 1, 128, 1, ''),
    ('upsample\n$\\times$2',       1, 128, 2, 'k=2'),
    ('up[1]\n3$\\times$ ResBlock', 2, 64,  2, ''),
    ('upsample\n$\\times$2',       2, 64,  4, 'k=2'),
    ('up[0]\n3$\\times$ ResBlock', 3, 32,  4, ''),
    ('norm_out + swish',           3, 32,  4, ''),
    ('conv_out\n3$\\times$3',      3, 1,   4, ''),
]
stage_color = {0: '#a6cee3', 1: '#fdae61', 2: '#fdae61', 3: '#a6d96a'}
stage_label = {0: 'S0  (latent res)', 1: 'S1  (up to 2$\\times$)',
               2: 'S2  (up to 4$\\times$)', 3: 'S3  (final)'}

fig, ax = plt.subplots(figsize=(15, 4.6))
box_w, box_h, gap = 1.1, 1.4, 0.25
xs = []
for i, (label, st, c_out, scale, note) in enumerate(layers):
    x = i * (box_w + gap); xs.append(x)
    ax.add_patch(mpatches.FancyBboxPatch(
        (x, 0), box_w, box_h, boxstyle='round,pad=0.02',
        fc=stage_color[st], ec='black', lw=1.0))
    ax.text(x + box_w/2, box_h/2 + 0.05, label, ha='center', va='center', fontsize=8.5)
    if note:
        ax.text(x + box_w/2, box_h - 0.18, note, ha='center', va='top',
                fontsize=7, color='#a64600', style='italic')
    ax.text(x + box_w/2, -0.18, f'$\\sigma$ = {scale}', ha='center', va='top', fontsize=8, color='#444')
    ax.text(x + box_w/2, -0.42, f'$C$ = {c_out}',     ha='center', va='top', fontsize=8, color='#444')
    if i < len(layers) - 1:
        ax.annotate('', xy=(x + box_w + gap, box_h/2), xytext=(x + box_w, box_h/2),
                    arrowprops=dict(arrowstyle='-|>', lw=1, color='gray'))
by_stage = {}
for i, (_, st, *_) in enumerate(layers): by_stage.setdefault(st, []).append(i)
for st, idxs in by_stage.items():
    x0 = xs[idxs[0]]; x1 = xs[idxs[-1]] + box_w
    ax.plot([x0, x0, x1, x1], [box_h+0.18, box_h+0.34, box_h+0.34, box_h+0.18],
            color='black', lw=1.0)
    ax.text((x0+x1)/2, box_h + 0.55, stage_label[st], ha='center', fontsize=10, weight='bold')
total_w = len(layers) * box_w + (len(layers) - 1) * gap
ax.set_xlim(-0.4, total_w + 0.4); ax.set_ylim(-1.0, box_h + 1.0); ax.set_axis_off()
ax.set_title('Toy 2D VAE decoder: layer sequence, channel counts $C$, spatial scale $\\sigma$', fontsize=11)
fig.tight_layout(); plt.show()

In [ ]:
# Stage flow: latent -> N+1 CPU buffers, with per-stage delta_r, k, output channels & scale.
def stage_channels(cfg):
    cs = [cfg.ch * cfg.ch_mult[-1]]
    for i in reversed(range(cfg.num_resolutions)):
        cs.append(cfg.ch * cfg.ch_mult[i])
    cs[-1] = cfg.out_channels
    return cs

L_lat_demo = 32
Cs = stage_channels(vae_cfg)
fig, ax = plt.subplots(figsize=(11.5, 4.6))
n = vae_cfg.num_resolutions + 1
xs = np.arange(n + 1) * 2.6
ax.add_patch(mpatches.FancyBboxPatch((xs[0]-0.7, 1.4), 1.4, 1.0,
    boxstyle='round,pad=0.05', fc='#dfdfdf', ec='black'))
ax.text(xs[0], 1.9, f'$\\bf{{z}}$\n${vae_cfg.z_dim}\\times {L_lat_demo}^d$',
        ha='center', va='center', fontsize=10)
stage_color = {0: '#a6cee3', 1: '#fdae61', 2: '#fdae61', 3: '#a6d96a'}
for s in range(n):
    side = L_lat_demo * scales[s]
    ax.add_patch(mpatches.FancyBboxPatch((xs[s+1]-0.7, 1.4), 1.4, 1.0,
        boxstyle='round,pad=0.05', fc=stage_color.get(min(s,3), '#a6d96a'), ec='black'))
    ax.text(xs[s+1], 1.9, f'$h_{s}$\n${Cs[s]}\\times {side}^d$',
            ha='center', va='center', fontsize=10)
    ax.annotate('', xy=(xs[s+1]-0.7, 1.9), xytext=(xs[s]+0.7, 1.9),
                arrowprops=dict(arrowstyle='-|>', color='black', lw=1.5))
    k_ = scales[s] // (scales[s-1] if s>0 else 1)
    ax.text((xs[s]+xs[s+1])/2, 2.7, f'S{s}\n$\\delta_r$={deltas[s]}, k={k_}',
            ha='center', fontsize=9, color='#444')
ax.text(xs[0], 0.6, 'latent', ha='center', fontsize=9, color='gray')
for s in range(n):
    kind = 'CPU buffer' if s < n-1 else 'CPU output'
    ax.text(xs[s+1], 0.6, kind, ha='center', fontsize=9, color='gray')
ax.text((xs[0]+xs[-1])/2, 0.0,
        'GPU at any moment: ONE tile of the CURRENT stage  ($\\sim T + 2 r_{\\max}$)',
        ha='center', fontsize=10, color='#b00020')
ax.set_xlim(-1.5, xs[-1]+1.5); ax.set_ylim(-0.5, 3.4); ax.set_axis_off()
ax.set_title(f'Multi-stage decoding: latent $\\to$ N+1 CPU buffers '
             f'(toy 2D VAE, $L_{{lat}} = {L_lat_demo}$)', fontsize=12)
fig.tight_layout(); plt.show()

Per-stage $\delta_r \in \{5, 6, 6, 7\}$, much smaller than the cumulative 24, and *each is paid against the stage's own source resolution*. Stage 0 reads with a 5-cell halo at latent resolution; stage 3 reads with a 7-cell halo at $4\times$ latent resolution. The halo never grows out of the network's natural scale.

## 7. Per-stage mechanics

A stage with upsample factor $k\in\{1,2\}$, source scale $\sigma_{\mathrm{in}}$, destination scale $\sigma_{\mathrm{out}} = k\,\sigma_{\mathrm{in}}$:

1. **Read** $\left.x\right|_{\Omega'^{+r}}$ from the source buffer. In source-buffer coordinates that is $\sigma_{\mathrm{in}}\cdot\Omega'^{+r}$.
2. **Compute** $f$. Output extent per axis = $k\,(|\Omega'|+2r)$.
3. **Crop** $k\cdot r$ cells from each side; keep the central $k\,|\Omega'|$.
4. **Write** to the destination buffer at position $\sigma_{\mathrm{out}}\cdot \Omega'$.

`compute_read_window` and `compute_crop_spec` implement steps 1 and 3.

In [ ]:
from diffsci2.extra.chunk_decode_2 import TileSpec, compute_read_window, compute_crop_spec

tile = TileSpec(ranges=((8, 16), (8, 16)))   # 8x8 latent tile starting at (8, 8)
spatial = (32, 32)
periodic = (False, False)
for stage_local_lat, src_scale, dest_scale in [
    (5, 1, 1),  # stage 0 (no upsample)
    (6, 1, 2),  # first upsampling stage
    (6, 2, 4),  # second upsampling stage
    (7, 4, 4),  # final (no upsample)
]:
    rw = compute_read_window(tile, stage_local_lat, src_scale, spatial, periodic)
    up = dest_scale // src_scale
    cs = compute_crop_spec(tile, rw, src_scale, dest_scale, up, spatial, periodic)
    print(f'stage_local_lat={stage_local_lat}, src_scale={src_scale}, dest_scale={dest_scale}, k={up}')
    print(f'  read window (latent units):  {rw.lat_ranges}')
    print(f'  read window (source units):  {rw.src_ranges}')
    print(f'  output crop slices:          {cs.tile_slices}')
    print(f'  destination ranges (output): {cs.dest_ranges}\n')

In [ ]:
# A single tile through one upsampling stage (k=2). Both panels at the same physical scale.
L_lat, r, k = 16, 3, 2
cs, ce = 4, 12; L_dst = L_lat * k
rs_lat, re_lat = max(0, cs - r), min(L_lat, ce + r)
fig, axes = plt.subplots(1, 2, figsize=(11, 5.5))

ax = axes[0]
ax.set_xlim(-1, L_lat + 1); ax.set_ylim(L_lat + 1, -1); ax.set_aspect('equal')
ax.set_xticks([]); ax.set_yticks([])
ax.set_title(f'Input  (source, {L_lat}$\\times${L_lat} latent cells)')
ax.add_patch(mpatches.Rectangle((-0.5, -0.5), L_lat, L_lat, fill=False, ec='black', lw=1.2))
ax.add_patch(mpatches.Rectangle((rs_lat-0.5, rs_lat-0.5), re_lat-rs_lat, re_lat-rs_lat,
    facecolor='#ffd9b3', edgecolor='#cc5500', lw=2, alpha=0.8,
    label=r"read window  $\Omega'^{+r}$"))
ax.add_patch(mpatches.Rectangle((cs-0.5, cs-0.5), ce-cs, ce-cs,
    facecolor='#a6d96a', edgecolor='#1a9850', lw=2, label=r"valid tile  $\Omega'$"))
for j in range(L_lat + 1):
    ax.plot([j-0.5, j-0.5], [-0.5, L_lat-0.5], color='gray', lw=0.25)
    ax.plot([-0.5, L_lat-0.5], [j-0.5, j-0.5], color='gray', lw=0.25)

ax = axes[1]
ax.set_xlim(-2, L_dst + 2); ax.set_ylim(L_dst + 2, -2); ax.set_aspect('equal')
ax.set_xticks([]); ax.set_yticks([])
ax.set_title(f'Output (dest, {L_dst}$\\times${L_dst}, scale $k\\sigma_{{in}}$, k = {k})')
ax.add_patch(mpatches.Rectangle((-0.5, -0.5), L_dst, L_dst, fill=False, ec='black', lw=1.2))
ow0, ow1 = rs_lat * k, re_lat * k
ax.add_patch(mpatches.Rectangle((ow0-0.5, ow0-0.5), ow1-ow0, ow1-ow0,
    facecolor='#ffd9b3', edgecolor='#cc5500', lw=1.5, alpha=0.45,
    label=fr"$f$ on read window  ($k(|\Omega'|+2r) = {ow1-ow0}$ px)"))
cs_dst, ce_dst = cs * k, ce * k
ax.add_patch(mpatches.Rectangle((cs_dst-0.5, cs_dst-0.5), ce_dst-cs_dst, ce_dst-cs_dst,
    facecolor='#a6d96a', edgecolor='#1a9850', lw=2,
    label=fr"kept  ($k|\Omega'| = {ce_dst-cs_dst}$ px)"))
for j in range(L_dst + 1):
    lw = 0.25 if (j % k) else 0.6
    ax.plot([j-0.5, j-0.5], [-0.5, L_dst-0.5], color='gray', lw=lw)
    ax.plot([-0.5, L_dst-0.5], [j-0.5, j-0.5], color='gray', lw=lw)

handles, labels_ = [], []
for ax_ in axes:
    for h, l in zip(*ax_.get_legend_handles_labels()):
        if l not in labels_: handles.append(h); labels_.append(l)
fig.legend(handles, labels_, loc='lower center', ncol=2, fontsize=9,
           bbox_to_anchor=(0.5, -0.02), frameon=False)
fig.suptitle('A single tile through one upsampling stage (k = 2):  read with halo, '
             'compute, crop $k\\cdot r$ pixels per side, write', fontsize=12)
fig.tight_layout(rect=[0, 0.08, 1, 1]); plt.show()

## 8. Sweeping the latent

We partition $\Omega$ into tiles whose *valid centres* are non-overlapping and tile $\Omega$.

$$
\mathrm{valid\_per\_tile} = \mathtt{chunk\_latent} - 2\, r_0
$$

where $r_0$ is the stage-0 RF radius. Boundary tiles are clamped (default) or periodically wrapped. `make_center_spans_1d` is the implementation.

In [ ]:
L_demo = 32
r0 = radii[0]
for chunk_latent_ in [16, 20, 24, 32]:
    spans_ = cd2.make_center_spans_1d(L_demo, chunk_latent_, r0)
    valid = max(1, chunk_latent_ - 2 * r0)
    print(f'  chunk_latent={chunk_latent_:2d}  valid={valid:2d}  ->  {len(spans_)} tile(s):  {spans_}')

In [ ]:
# 1D sweep visual: read windows on top, valid centres below.
def draw_sweep(L, chunk, r):
    spans_ = cd2.make_center_spans_1d(L, chunk, r)
    fig, ax = plt.subplots(figsize=(10, 2.6))
    ax.set_xlim(-r - 1, L + r + 1); ax.set_ylim(2.5, -0.4)
    ax.set_yticks([0.5, 1.7]); ax.set_yticklabels(['valid centres', 'read windows'])
    ax.set_xlabel('axis (latent cells)')
    ax.set_title(f'chunk_latent = {chunk}, halo r = {r}, L = {L}.   '
                 f'{len(spans_)} tile(s),   valid_per_tile $\\approx$ {max(1, chunk - 2*r)}')
    ax.add_patch(mpatches.Rectangle((-0.5, 0.1), L, 0.8, fill=False, ec='black'))
    colors = plt.cm.tab10(np.linspace(0, 1, max(len(spans_), 2)))
    for k, ((s, e), c) in enumerate(zip(spans_, colors)):
        rs = max(0, s - r); re = min(L, e + r)
        ax.add_patch(mpatches.Rectangle((rs - 0.5, 1.3), re - rs, 0.8,
            facecolor=c, alpha=0.35, edgecolor=c, lw=1.5))
        ax.add_patch(mpatches.Rectangle((s - 0.5, 1.3), e - s, 0.8,
            facecolor=c, alpha=0.85, edgecolor='black', lw=1))
        ax.add_patch(mpatches.Rectangle((s - 0.5, 0.1), e - s, 0.8,
            facecolor=c, alpha=0.85, edgecolor='black', lw=1))
        ax.text((s + e)/2 - 0.5, 0.5, f'{k}', ha='center', va='center', color='white',
                fontweight='bold', fontsize=8)
        ax.text((s + e)/2 - 0.5, 1.7, f'{k}', ha='center', va='center', color='white',
                fontweight='bold', fontsize=8)
    fig.tight_layout(); return fig

draw_sweep(L=32, chunk=24, r=5); plt.show()
draw_sweep(L=32, chunk=12, r=5); plt.show()

In [ ]:
# 2D sweep on a real Berea encoding. Lime: valid tiles in pixel space.
# Right: latent grid with one tile's halo highlighted.
patch_2d = torch.from_numpy(berea_slice[:256, :256][None, None]).to(DEVICE).float()
with torch.no_grad():
    z2 = vae.encode(patch_2d, sample=False)['zsample']
L_lat2 = z2.shape[-1]
chunk_latent_2d = 32; r0 = radii[0]
spans_2d = cd2.make_center_spans_1d(L_lat2, chunk_latent_2d, r0)
F2 = patch_2d.shape[-1] // L_lat2

fig, axes = plt.subplots(1, 2, figsize=(11, 5.4))
axes[0].imshow(patch_2d[0, 0].cpu().numpy(), cmap='gray')
for hs, he in spans_2d:
    for ws, we in spans_2d:
        axes[0].add_patch(mpatches.Rectangle(
            (ws*F2-0.5, hs*F2-0.5), (we-ws)*F2, (he-hs)*F2,
            fill=False, edgecolor='lime', lw=1.5))
axes[0].set_title('input patch (256$\\times$256 px)\nvalid tiles (lime)')
axes[0].set_xticks([]); axes[0].set_yticks([])

ax = axes[1]
ax.imshow(z2[0, 0].cpu().numpy(), cmap='viridis',
          extent=[-0.5, L_lat2-0.5, L_lat2-0.5, -0.5])
for hs, he in spans_2d:
    for ws, we in spans_2d:
        ax.add_patch(mpatches.Rectangle((ws-0.5, hs-0.5), we-ws, he-hs,
                                         fill=False, edgecolor='white', lw=1.4))
hs, he = spans_2d[0]; ws, we = spans_2d[1]
rh0, rh1 = max(0, hs-r0), min(L_lat2, he+r0)
rw0, rw1 = max(0, ws-r0), min(L_lat2, we+r0)
ax.add_patch(mpatches.Rectangle((rw0-0.5, rh0-0.5), rw1-rw0, rh1-rh0,
    fill=False, edgecolor='red', lw=1.8, ls='--', label=f'halo for one tile  ($r_0$={r0})'))
ax.add_patch(mpatches.Rectangle((ws-0.5, hs-0.5), we-ws, he-hs,
    fill=False, edgecolor='red', lw=1.8, label="that tile's valid region"))
ax.set_title(f'latent $z_0$, {L_lat2}$\\times${L_lat2}\n'
             f'sweep: chunk_latent={chunk_latent_2d}, $r_0$={r0}, {len(spans_2d)**2} tiles')
ax.set_xticks([]); ax.set_yticks([])
ax.legend(loc='upper right', fontsize=8, framealpha=0.95)
fig.tight_layout(); plt.show()

## 9. GPU and CPU at scale

**At any instant.**
- *GPU:* one tile of activations of the current stage. Bounded by $C_{\max}\cdot(T+2r_{\max})^d$, **independent of $L$**.
- *CPU:* at most two stage buffers (source, destination). When stage $s+1$ is finished, buffer $s$ is freed.

**Buffer size.** Stage $s$ outputs a tensor of shape $(B,\, C_s,\, (L_\text{lat}\sigma_s)^d)$ in float32:
$$
\text{buffer}_s = 4 \cdot C_s \cdot (L_\text{lat}\sigma_s)^d \;\text{bytes.}
$$

In [ ]:
# Production VAE buffer accounting at L_pix = 1280, 2304.
prod_cfg = diffsci2.nets.VAENetConfig(
    dimension=3, in_channels=1, out_channels=1, z_channels=4, z_dim=4,
    ch=32, ch_mult=[1, 2, 4, 4], num_res_blocks=2, attn_resolutions=[],
    dropout=0.0, resolution=256, has_mid_attn=False, resamp_with_conv=True,
    attn_type='vanilla', tanh_out=False, input_bias=True, output_bias=True,
    with_time_emb=False, double_z=True, num_groups=32, patch_size=None,
    memory_efficient_variant=False, use_flash_attention=True, minimal_rf_mode=False,
)
F_prod = 8

def buffer_sizes(cfg, L_pix, ndim=3):
    L_lat_ = L_pix // F_prod
    _, sc = cd2.compute_stage_radii_and_scales(diffsci2.nets.VAENet(config=cfg).decoder)
    Cs = stage_channels(cfg)
    return [Cs[s] * (L_lat_ * sc[s]) ** ndim * 4 for s in range(cfg.num_resolutions + 1)]

for L_pix in [1280, 2304]:
    L_lat_ = L_pix // F_prod
    sizes = buffer_sizes(prod_cfg, L_pix)
    print(f'L_pix = {L_pix}  (L_lat = {L_lat_})')
    Cs = stage_channels(prod_cfg)
    _, sc = cd2.compute_stage_radii_and_scales(diffsci2.nets.VAENet(config=prod_cfg).decoder)
    for s in range(prod_cfg.num_resolutions + 1):
        print(f'  S{s}  C_s={Cs[s]:>4}  side={L_lat_*sc[s]:>5}  '
              f'{sizes[s]/2**30:>8.1f} GB')
    print()

In [ ]:
# Production buffer chart with the 1 TB CPU RAM line.
sizes_1280 = buffer_sizes(prod_cfg, L_pix=1280)
sizes_2304 = buffer_sizes(prod_cfg, L_pix=2304)
n_stages_p = max(len(sizes_1280), len(sizes_2304))
CPU_RAM_GB = 1024

fig, ax = plt.subplots(figsize=(8.5, 4.5))
bar_w = 0.35
for i, (name, sz) in enumerate([('$L_{pix}$ = 1280', sizes_1280),
                                 ('$L_{pix}$ = 2304', sizes_2304)]):
    padded = list(sz) + [np.nan] * (n_stages_p - len(sz))
    xs_ = np.arange(n_stages_p) + i * bar_w
    ax.bar(xs_, np.array(padded, dtype=float) / 2**30, width=bar_w, label=name)
ax.set_xticks(np.arange(n_stages_p) + bar_w / 2)
ax.set_xticklabels([f'S{s}' for s in range(n_stages_p)])
ax.set_ylabel('stage buffer size (GB, log)'); ax.set_yscale('log')
ax.set_title('Per-stage CPU buffer size --- production 3D VAE  '
             '(ch=32, ch_mult=[1,2,4,4], F=8)', fontsize=11)
ax.axhline(CPU_RAM_GB, ls='--', color='black', lw=1)
ax.text(0.02, CPU_RAM_GB * 1.18, 'CPU RAM (1 TB)',
        transform=ax.get_yaxis_transform(), fontsize=10)
ax.legend(fontsize=10, loc='upper left')
ax.grid(axis='y', alpha=0.3, which='both')
fig.tight_layout(); plt.show()

**Above the RAM line.** At $L_\text{pix}=2304$ a single stage buffer is $\sim 3$ TB --- past the 1 TB CPU RAM of our box. `chunk_decode_2` falls back to `MmapStageBuffer`, which presents the same interface as the in-memory `CPUStageBuffer` but is backed by a `numpy.memmap` on SSD. Reads and writes look like ordinary tensor indexing; the OS pages the file in and out transparently. Algorithm and correctness are unchanged. Pass `use_disk_offload=True` and `disk_offload_dir=...` to `chunk_decode_strategy_b`.

## 10. Normalisation consistency

Convs / ResBlocks / nonlinearities have finite RF. **GroupNorm** does not: it computes the mean and variance of each group across *all* spatial positions of its input. A tile and the global decoder pick *different* group statistics on the same convolutional output. Tiles processed independently produce visible *seams* at their boundaries.

Two clean fixes:

1. **Two-pass.** Run the decoder once just to gather the global statistics, then a second time to use them. Doubles the compute.
2. **First-tile cache.** Run the *first* tile of each stage normally and *cache* the group statistics it produced; force every other tile of the same stage to reuse those cached statistics. One pass, identical normalisation across tiles, no seams. `chunk_decode_2` uses (2).

In [ ]:
# Run the toy decoder both ways on the same Berea patch and compare.
patch = torch.from_numpy(berea_slice[:256, :256][None, None]).to(DEVICE).float()
with torch.no_grad():
    z = vae.encode(patch, sample=False)['zsample']
    x_mono = vae.decode(z, postdecode=False).cpu().numpy()[0, 0]

# (a) chunked, NO cached norms: per-tile GroupNorm stats -> seams.
x_chunk_raw = cd2.chunk_decode_2d(
    decoder, z, chunk_latent=16, use_cached_norms=False,
).cpu().numpy()[0, 0]

# (b) chunked, WITH cached norms: install CachedGroupNorm layers, then run.
decoder_cached = cd2.prepare_decoder_for_cached_decode(
    decoder, inplace=False).to(DEVICE).eval()
x_chunk_cached = cd2.chunk_decode_2d(
    decoder_cached, z, chunk_latent=16, use_cached_norms=True,
).cpu().numpy()[0, 0]

print('seam metric  max|chunk_raw - chunk_cached| =',
      f'{np.abs(x_chunk_raw - x_chunk_cached).max():.3f}')
print('first-tile bias  max|chunk_cached - mono|  =',
      f'{np.abs(x_chunk_cached - x_mono).max():.3f}')

In [ ]:
# Seam panel: monolithic / chunk no-cache / chunk cached / difference.
fig, axes = plt.subplots(1, 4, figsize=(15, 4))
vmin = float(min(x_mono.min(), x_chunk_raw.min(), x_chunk_cached.min()))
vmax = float(max(x_mono.max(), x_chunk_raw.max(), x_chunk_cached.max()))
axes[0].imshow(x_mono,         cmap='gray', vmin=vmin, vmax=vmax)
axes[0].set_title('monolithic decode\n(global GroupNorm stats)')
axes[1].imshow(x_chunk_raw,    cmap='gray', vmin=vmin, vmax=vmax)
axes[1].set_title('chunk, no cached norms\n(per-tile stats $\\Rightarrow$ seams)')
axes[2].imshow(x_chunk_cached, cmap='gray', vmin=vmin, vmax=vmax)
axes[2].set_title('chunk, cached norms\n(shared stats $\\Rightarrow$ no seams)')
diff = x_chunk_raw - x_chunk_cached
im = axes[3].imshow(diff, cmap='RdBu_r',
                     vmin=-np.abs(diff).max(), vmax=np.abs(diff).max())
axes[3].set_title('chunk_raw $-$ chunk_cached\n(the seam pattern)')
for ax in axes: ax.set_xticks([]); ax.set_yticks([])
fig.colorbar(im, ax=axes[3], fraction=0.046)
fig.suptitle('Tile boundaries every chunk_latent$\\times$F = 64 px. '
             'Cached norms remove the seam.', y=1.04, fontsize=12)
fig.tight_layout(); plt.show()

**Why first-tile caching is safe (here).** Micro-CT volumes are samples from a translation-invariant, ergodic measure. By the multivariate ergodic theorem, the empirical mean and variance computed over *any* sufficiently large sub-volume converge to the same population statistics --- including those of the *first* tile. The first-tile statistics are an estimator of the same population the global decoder would compute, modulo finite-tile noise. The seams disappear; the global bias relative to the monolithic decode is small in practice.

**The cleaner long-term fix.** Cached norms are a patch on existing GroupNorm checkpoints. Architecturally, the right answer is to retrain without globally-aware normalisation: channel LayerNorm, RMSNorm, instance-on-window, or no normalisation at all. All of those have finite (in fact zero) effective radius, and chunk decoding becomes bit-exact end-to-end --- no first-tile cache, no ergodicity argument, no bias to absorb.

## 11. End-to-end

Encode a $512\times 512$ Berea patch to a $128\times 128$ latent and chunk-decode it back. `debug=1` prints per-stage summaries.

In [ ]:
patch_big = torch.from_numpy(berea_slice[None, None]).to(DEVICE).float()
with torch.no_grad():
    z_big = vae.encode(patch_big, sample=False)['zsample']
    x_mono_big = vae.decode(z_big, postdecode=False).cpu().numpy()[0, 0]

x_chunk_big = cd2.chunk_decode_2d(
    decoder_cached, z_big, chunk_latent=32,
    use_cached_norms=True, debug=1,
).cpu().numpy()[0, 0]

print(f'\nmax|chunk_cached - mono| = {np.abs(x_chunk_big - x_mono_big).max():.3f}')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 4.6))
axes[0].imshow(berea_slice, cmap='gray'); axes[0].set_title('Berea slice (input, 512$\\times$512)')
axes[1].imshow(x_mono_big,  cmap='gray'); axes[1].set_title('monolithic VAE decode')
axes[2].imshow(x_chunk_big, cmap='gray'); axes[2].set_title('chunk-decoded\n(cached norms, chunk_latent=32)')
for ax in axes: ax.set_xticks([]); ax.set_yticks([])
fig.tight_layout(); plt.show()

**Production extras.**
- `MmapStageBuffer` --- disk-backed stage buffers when CPU RAM is insufficient. Pass `use_disk_offload=True`, `disk_offload_dir=...`.
- `chunk_decode_parallel` --- multi-GPU dispatch. Tiles are processed concurrently on replicated decoder copies; cached-norm statistics computed on device 0 are synced to all other devices via `sync_cached_norms_across_devices`. Speedup is roughly linear in the number of GPUs when the per-stage tile count is large.

## 12. Take-aways

- Tiled inference is *exact* for finite-RF networks, by identity (RF). Halos are paid in **latent units, one stage at a time**. Effective radius is computable layer by layer: $r(f\circ g)=r(f)+r(g)$, $r(f+g)=\max(r(f), r(g))$.
- Peak GPU memory becomes $\mathcal{O}\!\bigl((T+2r_{\max})^d\bigr)$, **independent of $L$**. Peak CPU is two adjacent stage buffers; SSD-backed mmap takes over above the RAM line.
- GroupNorm is the only globally-aware operator left in the existing decoder. The first-tile cache makes the algorithm work *because the data is ergodic*; the architectural fix is to retrain without globally-aware normalisation, then drop the cache entirely.
- Implementation: `diffsci2.extra.chunk_decode_2.chunk_decode_strategy_b` (single-GPU), `chunk_decode_parallel` (multi-GPU), both dimension-agnostic (2D and 3D), with optional disk offload.